# Ayudantía 1
### IMT2112: Algoritmos Paralelos para Computación Científica
Profesor: Ignacio Labarca (ignacio.labarca@uc.cl) \
Ayudantes: Javiera Sanhueza (jsanhuezp@uc.cl) & Juan Küster (juan.kuster@uc.cl)

In [118]:
import numpy as np
import time as tm
import pandas as pd

## Funciones de joblib

### 1. Inicialización
Creamos un _pool_ de trabajadores que harán las tareas en paralelo mediante la clase ```Parallel```.





In [119]:
from joblib import Parallel

In [120]:
parallel_pool = Parallel(n_jobs = 2)
print(parallel_pool)

Parallel(n_jobs=2)


In [121]:
print("The number of workers is:", parallel_pool.n_jobs)

The number of workers is: 2


Una forma de asignarles una tarea específica a cada trabajador, es mediante la función o _decorator_ ```delayed```. Esta retorna una variante paralelizada de la función entregada para ```joblib```.

In [122]:
from joblib import delayed

In [123]:
parallel_sqrt = delayed(np.sqrt)

Antes de asignar la función a los _workers_, le entregamos las variables de input para la función. Si bien hay que especificar cada tarea, ```joblib``` se encarga de asignarlas entre los _workers_.

In [124]:
parallel_tasks = [parallel_sqrt(i) for i in range(10)]

In [125]:
array = [i for i in range(10)]
parallel_tasks_2 = map(parallel_sqrt, array)

Ahora, le entregamos al _pool_ de trabajadores paralelizados, todas las tareas.

In [126]:
parallel_results = parallel_pool(parallel_tasks)
parallel_results_2 = parallel_pool(parallel_tasks_2)

In [127]:
print(parallel_results)

[np.float64(0.0), np.float64(1.0), np.float64(1.4142135623730951), np.float64(1.7320508075688772), np.float64(2.0), np.float64(2.23606797749979), np.float64(2.449489742783178), np.float64(2.6457513110645907), np.float64(2.8284271247461903), np.float64(3.0)]


In [128]:
print(parallel_results_2)

[np.float64(0.0), np.float64(1.0), np.float64(1.4142135623730951), np.float64(1.7320508075688772), np.float64(2.0), np.float64(2.23606797749979), np.float64(2.449489742783178), np.float64(2.6457513110645907), np.float64(2.8284271247461903), np.float64(3.0)]


In [129]:
parallel_results == parallel_results_2

True

Crearemos un nuevo _pool_, pero con el máximo de _workers_ disponibles.

In [130]:
from joblib import cpu_count
print("Number of cores found by joblib:", cpu_count())

Number of cores found by joblib: 2


In [131]:
parallel_pool = Parallel(n_jobs = -1)

In [132]:
print(parallel_pool.n_jobs)

-1


Podemos reutilizar el mismo _pool_ de _workers_, lo que suele ser más rápido dado que ya está inicializado. Creamos la función potencia cuadrada.

In [133]:
def squared(n):
  return n**2
parallel_square = delayed(squared)

In [134]:
parallel_tasks_square = [parallel_square(i) for i in array]
parallel_results_square = parallel_pool(parallel_tasks_square)
print(parallel_results_square)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


In [135]:
parallel_tasks_root = [parallel_sqrt(i) for i in parallel_results_square]
parallel_results_root = parallel_pool(parallel_tasks_root)
print(parallel_results_root)

[np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0)]


Para obtener más información sobre cada iteración, utilizamos ```verbose```.

In [136]:
tasks = [delayed(np.sqrt)(i) for i in range(100000)]
with Parallel(n_jobs=2, verbose=20) as parallel_pool:
    parallel_results = parallel_pool(tasks)

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done      1 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=2)]: Batch computation too fast (0.005181789398193359s.) Setting batch_size=2.
[Parallel(n_jobs=2)]: Done      2 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done      3 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done      4 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done      6 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=2)]: Batch computation too fast (0.0018575191497802734s.) Setting batch_size=4.
[Parallel(n_jobs=2)]: Done      8 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done     10 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done     12 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done     16 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=2)]: Batch computation too fast (0.0039501190185546875s.) Setting batch_size=8.
[Parallel(n_jobs=2)]: Done     2

Vemos que ```joblib``` ajusta el ```batch_size``` automáticamente. El _batch size_ es la cantidad de tareas asignadas dinámicamente a cada _worker_. Por default, el tamaño del _batch_ es 1. Dado que se puede caer en un _overhead_ al asignar una nueva tarea, se puede evitar fijando la cantidad de tareas dadas simultáneamente a cada _worker_. Es decir, especificando un _batch size_.

In [137]:
with Parallel(n_jobs=2, verbose=10, batch_size=50000) as parallel_pool:
    parallel_results = parallel_pool(tasks)

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   5003 out of 100000 | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done  20000 out of 100000 | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done  44995 out of 100000 | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done  69990 out of 100000 | elapsed:    0.4s
[Parallel(n_jobs=2)]: Done 100000 out of 100000 | elapsed:    0.5s finished


El último atributo de ```Parallel``` que veremos es el ```backend```. Joblib tiene varios _backends_ de paralelización:


*   ```loky```: basado en el modelo de _multiprocessing_, es decir, crea diferentes procesos y asigna tareas a cada proceso. Es el default.
*   ```threading```: basado en el modelo de _multithreading_, es decir, crea diferentes _threads_ dentro de un mismo proceso.
* ```sequential```: se usa para forzar una ejecución secuencial en lugar de paralelizada.

A priori, _multiprocessing_ tiene mayor _overhead_ que _multithreading_, pero es más robusto cuando no hay _race conditions_.

La eficiencia de cada _backend_ depende fuertemente de la arquitectura del computador y las tareas a realizar.


In [138]:
with Parallel(n_jobs=4, batch_size=1000, verbose=10, backend='loky') as parallel_pool:
    parallel_results = parallel_pool(tasks)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done      2 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done      8 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done   5008 out of 100000 | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done  10008 out of 100000 | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done  17008 out of 100000 | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done  24008 out of 100000 | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done  33008 out of 100000 | elapsed:    0.3s
[Parallel(n_jobs=4)]: Done  42008 out of 100000 | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done  53008 out of 100000 | elapsed:    0.5s
[Parallel(n_jobs=4)]: Done  64008 out of 100000 | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done  77008 out of 100000 | elapsed:    0.8s
[Parallel(n_jobs=4)]: Done  90008 out of 100000 | elapsed:    0.9s
[Parallel(n_jobs=4)]: Done  96899 out of 100000 | elapsed:    0.9s
[Parallel(n_jobs=4)]: Done  98384 out of 100000 | elap

In [139]:
with Parallel(n_jobs=4, batch_size=1000, verbose=10, backend='threading') as parallel_pool:
    parallel_results = parallel_pool(tasks)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done      2 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done      8 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done   5008 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done  10008 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done  17008 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done  24008 out of 100000 | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done  33008 out of 100000 | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done  42008 out of 100000 | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done  53008 out of 100000 | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done  64008 out of 100000 | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done  77008 out of 100000 | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done  90008 out of 100000 | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done  96899 out of 100000 | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done  98384 out of 100000 |

También se puede "sugerir" en lugar de espcificar el _backend_ de forma explícita, usando ```prefer = 'processes'``` o ```prefer = 'threading'``` en su lugar.

### 2. Variables globales

Para leer una variable global, también es necesario definir una función para leerla.

In [140]:
MY_GLOBAL_VAR = 3.4

def return_global_var():
  return MY_GLOBAL_VAR

In [141]:
n_workers = 2
tasks = [delayed(return_global_var)() for i in range(n_workers)]
with Parallel(n_jobs=n_workers, batch_size=1, verbose=10, backend='loky') as parallel_pool:
    parallel_results = parallel_pool(tasks)

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done 2 out of 2 | elapsed:    0.1s finished


In [142]:
print(parallel_results)

[3.4, 3.4]


Al entregarle la tarea de leer la variable, cada proceso va a crear su propia copia de la variable global. Por lo tanto, esta variable global no puede ser editada por los procesos individuales.

Definimos una función que nos permita sobreescribir una variable global.

In [143]:
def change_global_var(n):
    MY_GLOBAL_VAR = n
    return MY_GLOBAL_VAR

In [144]:
tasks = [delayed(change_global_var)(i) for i in range(n_workers)]
with Parallel(n_jobs=n_workers, batch_size=1, verbose=10, backend='loky') as parallel_pool:
    parallel_results = parallel_pool(tasks)

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done 2 out of 2 | elapsed:    0.0s finished


In [145]:
print(parallel_results)
print(MY_GLOBAL_VAR)

[0, 1]
3.4


Notamos que en realidad, no se está cambiando la variable global, sino la versión local de esta. Intentemos hacer una función que pueda cambiarla.

In [146]:
def add_to_global_var(n):
    MY_GLOBAL_VAR = MY_GLOBAL_VAR + n
    return MY_GLOBAL_VAR

In [147]:
tasks = [delayed(add_to_global_var)(i) for i in range(n_workers)]
with Parallel(n_jobs=n_workers, batch_size=1, verbose=10, backend='loky') as parallel_pool:
    parallel_results = parallel_pool(tasks)

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


UnboundLocalError: cannot access local variable 'MY_GLOBAL_VAR' where it is not associated with a value

Vemos que esto falla porque la función está intentando leer y escribir sobre la misma variable ```MY_GLOBAL_VAR```. Aquí el error es porque intentamos leer la variable global y escribir en la variable local al mismo tiempo.

### 4. Lectura databases

In [166]:
my_global_df = pd.DataFrame(data=np.arange(100), columns=["my_data"])

In [167]:
my_global_df

,my_data
0,0
1,1
2,2
3,3
4,4
...,...
95,95
96,96
97,97
98,98


In [168]:
def sum_data(start, end):
    my_local_data = my_global_df["my_data"][start:end]
    return np.sum(my_local_data)

In [169]:
chunk_size = my_global_df.shape[0] // n_workers
tasks = [delayed(sum_data)(i*chunk_size, (i+1)*chunk_size) for i in range(n_workers)]

In [170]:
with Parallel(n_jobs=n_workers, batch_size=1, verbose=10, backend='loky') as parallel_pool:
    parallel_results = parallel_pool(tasks)

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done 2 out of 2 | elapsed:    0.0s finished


In [171]:
print(parallel_results)

[((50,), np.int64(1225)), ((50,), np.int64(3725))]


Como tenemos 2 _workers_, el primer proceso suma todos los elementos de la primera mitad del database y el segundo proceso suma la segunda mitad. Notamos que ambos procesos tienen una copia del database completo, lo cual es ineficiente.

Podemos distribuir el database sobre los distintos procesos leyendo únicamente las partes que necesita cada proceso del database. Por ejemplo, que el _worker_ 1 solo lea la primera mitad y el otro lea solo la segunda mitad del database desde la memoria.

In [173]:
my_global_df.to_excel("my_database.xlsx", index=False)

In [179]:
def read_and_sum_data(start, end):
    my_local_df = pd.read_excel("my_database.xlsx", header=0, skiprows=range(1,start+1), nrows=end-start)
    return my_local_df.shape, np.sum(my_local_df["my_data"])

In [180]:
chunk_size = my_global_df.shape[0] // n_workers
tasks = [delayed(read_and_sum_data)(i*chunk_size, (i+1)*chunk_size) for i in range(n_workers)]

In [181]:
with Parallel(n_jobs=n_workers, batch_size=1, verbose=10, backend='loky') as parallel_pool:
    parallel_results = parallel_pool(tasks)

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done 2 out of 2 | elapsed:    0.0s finished


In [182]:
print(parallel_results)

[((50, 1), np.int64(1225)), ((50, 1), np.int64(3725))]


Vemos que las sumas están bien y que los dataframes locales tienen la mitad del tamaño del archivo original.

## Actividad

Defina una función que sume dos elementos dados.
Ejecute para el array la suma entre los array [i for i in range(1, 11)] y [i for i in range(5, 16)], con ```n_jobs = 2``` y ```verbose = 10```. Observe que sucede con su ```batch_size```.

In [185]:
def suma(x,y):
  return x + y

parallel_sum = delayed(suma)

In [183]:
A =  [i for i in range(1, 11)]
B = [i for i in range(5, 16)]

In [186]:
tasks = map(parallel_sum, A, B)

parallel_pool = Parallel(n_jobs = 2, verbose = 10)
parallel_results = parallel_pool(tasks)


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Batch computation too fast (0.002981424331665039s.) Setting batch_size=2.
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Batch computation too fast (0.0020465850830078125s.) Setting batch_size=4.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.0s finished


In [187]:
print(parallel_results)

[6, 8, 10, 12, 14, 16, 18, 20, 22, 24]
